# Health Risk - Data Preparation
Integrantes:
- Diego Fabrizio Mucha Alvarez
- Ivan Ruben Cunyas Ramos
- Mauricio Andrés Canchis Fernández
- Juan Jose Rodriguez Velásquez
- María Laura Aragon Flores

En este notebook se realizará la fase 3 de CRISP-DM que es el data preparation. Para esta fase vamos a usar lo que se sugirio en el EDA y hacer la preparación de nuestra data para los modelos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 0. Carga y Configuración Inicial del Dataset

### 0.1. Cargar Dataset

In [ ]:
df_raw = pd.read_csv("../data/raw/SEER_data.csv")

/tmp/ipykernel_204435/1286513731.py:1: DtypeWarning: Columns (0: Regional nodes examined (1988+), 1: Regional nodes positive (1988+)) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("../data/raw/SEER_data.csv")


### 0.2. Eliminar columnas repetidas

In [ ]:
columns_to_drop = [
    "Tumor Size Summary (2016+)",
    "Tumor Size Over Time Recode (1988+)",
    "Race recode (White, Black, Other)",
    "Summary stage 2000 (1998-2017)",
    "Primary Site - labeled",
    "SEER historic stage A (1973-2015)",
    "CS lymph nodes (2004-2015)",
    "CS mets at dx (2004-2015)",
    "Survival Days",
    "Diagnostic Confirmation",
    "Laterality",
    "Regional nodes examined (1988+)",
    "Behavior recode for analysis",
    "Reason no cancer-directed surgery",
    "SEER cause-specific death classification",
    "Vital status recode (study cutoff used)",
    "Radiation recode",
    "Regional nodes positive (1988+)",
    "CS extension (2004-2015)"
]

df_raw = df_raw.drop(columns=columns_to_drop)

### 0.3. Renombrar columnas

In [ ]:
new_column_names = [
    'age_group',
    'sex',
    'diagnosis_year',
    'histologic_type',
    'cancer_stage',
    'cancer_site',
    'tumor_grade',
    'bone_metastasis_at_dx',
    'brain_metastasis_at_dx',
    'liver_metastasis_at_dx',
    'lung_metastasis_at_dx',
    'primary_site_surgery',
    'received_chemotherapy',
    'survival_months',
]

df_raw.columns = new_column_names

### 0.4. Filtrar fecha de las variables

In [ ]:
df_raw = df_raw[df_raw["diagnosis_year"] >= 2010]

## 1. Análisis Inicial de los Datos

Se eliminan variables nulas y duplicadas

In [ ]:
df_raw = df_raw.dropna()
df_raw = df_raw.drop_duplicates()

Encoding de la variable age_group

In [ ]:
dic_encoding = {"00 years":0, "01-04 years":1, "05-09 years": 2, "10-14 years": 3, "15-19 years": 4, "20-24 years": 5, "25-29 years": 6, "30-34 years": 7, "35-39 years": 8, "40-44 years": 9, "45-49 years": 10, "50-54 years": 11, "55-59 years": 12, "60-64 years": 13, "65-69 years": 14}
df_raw["age_group"] = df_raw["age_group"].map(dic_encoding)

Cambiar tipo de dato a int a la variable survival_months, primero vamos a eliminar todas los samples que tienen a esa variable con la clase unknown.

In [ ]:
df_raw = df_raw[df_raw["survival_months"] != "Unknown"]
df_raw["survival_months"] = df_raw["survival_months"].astype(int)

Categorizar los valores de la columna primary_site_surgery
- 00 **→** No surgery
- 01-29 **→** Local/Minor surgery
- 30-59 **→** Partial/Organ surgery
- 60-89 **→** Extensive/Radical surgery
- 90 **→** Surgery, NOS
- 98 **→** Not applicable
- 99 **→** Unkown

In [ ]:
df_raw.loc[df_raw["primary_site_surgery"] == "Blank(s)", "primary_site_surgery"] = "Unknown"

def map_surgery(code):
    if str(code) == "Unknown" or str(code) == "Blank(s)":
        return "Unknown"
    code = int(code)
    if code == 0:
        return "no_surgery"
    elif 1 <= code <= 29:
        return "minor_surgery"
    elif 30 <= code <= 59:
        return "partial_surgery"
    elif 60 <= code <= 89:
        return "radical_surgery"
    elif code == 90:
        return "surgery_nos"
    elif code == 98:
        return "not_applicable"
    elif code == 99:
        return "Unknown"
    else:
        print(code)
        return "other"  # safety fallback

df_raw["primary_site_surgery"] = df_raw["primary_site_surgery"].apply(map_surgery)

Agrupar los tipos de cancer en categorias de la columna cancer_site

In [ ]:
site_group_map = {
    # Cancer de cabeza y Cuello
    "Lip": "Head and Neck",
    "Tongue": "Head and Neck",
    "Salivary Gland": "Head and Neck",
    "Floor of Mouth": "Head and Neck",
    "Gum and Other Mouth": "Head and Neck",
    "Nasopharynx": "Head and Neck",
    "Tonsil": "Head and Neck",
    "Oropharynx": "Head and Neck",
    "Hypopharynx": "Head and Neck",
    "Other Oral Cavity and Pharynx": "Head and Neck",

    # CAncer de sistema digestivo
    "Esophagus": "Digestive",
    "Stomach": "Digestive",
    "Small Intestine": "Digestive",
    "Cecum": "Digestive",
    "Appendix": "Digestive",
    "Ascending Colon": "Digestive",
    "Hepatic Flexure": "Digestive",
    "Transverse Colon": "Digestive",
    "Splenic Flexure": "Digestive",
    "Descending Colon": "Digestive",
    "Sigmoid Colon": "Digestive",
    "Large Intestine, NOS": "Digestive",
    "Rectosigmoid Junction": "Digestive",
    "Rectum": "Digestive",
    "Anus, Anal Canal and Anorectum": "Digestive",
    "Liver": "Digestive",
    "Intrahepatic Bile Duct": "Digestive",
    "Gallbladder": "Digestive",
    "Other Biliary": "Digestive",
    "Pancreas": "Digestive",
    "Retroperitoneum": "Digestive",
    "Peritoneum, Omentum and Mesentery": "Digestive",
    "Other Digestive Organs": "Digestive",

    # Cancer respiratorio
    "Nose, Nasal Cavity and Middle Ear": "Respiratory",
    "Larynx": "Respiratory",
    "Lung and Bronchus": "Respiratory",
    "Pleura": "Respiratory",
    "Trachea, Mediastinum and Other Respiratory Organs": "Respiratory",

    # Otros tumores solidos
    "Bones and Joints": "Bone and Soft Tissue",
    "Soft Tissue including Heart": "Bone and Soft Tissue",
    "Melanoma of the Skin": "Skin",
    "Other Non-Epithelial Skin": "Skin",
    "Breast": "Breast",

    # Genitales femeninos
    "Cervix Uteri": "Female Genital",
    "Corpus Uteri": "Female Genital",
    "Uterus, NOS": "Female Genital",
    "Ovary": "Female Genital",
    "Vagina": "Female Genital",
    "Vulva": "Female Genital",
    "Other Female Genital Organs": "Female Genital",

    # Genitales masculinos
    "Prostate": "Male Genital",
    "Testis": "Male Genital",
    "Penis": "Male Genital",
    "Other Male Genital Organs": "Male Genital",

    # Urinarios
    "Urinary Bladder": "Urinary",
    "Kidney and Renal Pelvis": "Urinary",
    "Ureter": "Urinary",
    "Other Urinary Organs": "Urinary",

    # Otros sistemas mayores
    "Eye and Orbit": "Eye",
    "Brain and Other Nervous System": "Brain/CNS",
    "Thyroid": "Endocrine",
    "Other Endocrine including Thymus": "Endocrine",

    # Hematologicos o raros
    "Hodgkin - Nodal": "Hematologic",
    "Hodgkin - Extranodal": "Hematologic",
    "NHL - Nodal": "Hematologic",
    "NHL - Extranodal": "Hematologic",
    "Myeloma": "Hematologic",
    "Acute Lymphocytic Leukemia": "Hematologic",
    "Chronic Lymphocytic Leukemia": "Hematologic",
    "Other Lymphocytic Leukemia": "Hematologic",
    "Acute Myeloid Leukemia": "Hematologic",
    "Acute Monocytic Leukemia": "Hematologic",
    "Chronic Myeloid Leukemia": "Hematologic",
    "Other Myeloid/Monocytic Leukemia": "Hematologic",
    "Other Acute Leukemia": "Hematologic",
    "Aleukemic, subleukemic and NOS": "Hematologic",
    "Mesothelioma": "Hematologic",
    "Kaposi Sarcoma": "Hematologic",
    "Miscellaneous": "Hematologic",

    # No se sabe o no validos
    "Invalid": "Unknown/Invalid"
}

df_raw["cancer_site"] = df_raw["cancer_site"].map(site_group_map).fillna("Other/Unmapped")

Convertir las 4 columnas de metastasis a una sola para verficar si hay metastasis o no

In [ ]:
met_cols = [
    "bone_metastasis_at_dx",
    "brain_metastasis_at_dx",
    "liver_metastasis_at_dx",
    "lung_metastasis_at_dx"
]
df_raw["metastasis_count"] = df_raw[met_cols].eq("Yes").sum(axis=1)
df_raw["metastasis_count"].value_counts()
df_raw = df_raw.drop(columns=["bone_metastasis_at_dx","brain_metastasis_at_dx", "liver_metastasis_at_dx", "lung_metastasis_at_dx"])

Convertir la variable de histologic_type a una categorica ya que los numeros reaclmente son codigos para tipos de histologia. Quedarse con los 15 primeros y el resto ponerlos diferentes

In [ ]:
top_histologies = df_raw["histologic_type"].value_counts().head(15).index
df_raw["histologic_type_grouped"] = df_raw["histologic_type"].where(df_raw["histologic_type"].isin(top_histologies), "Other")

Convertir la variable recieved_chemeotherapy a booleano

In [ ]:
df_raw["received_chemotherapy"] = df_raw["received_chemotherapy"].map({"Yes": 1, "No/Unknown": 0}).astype(int)

Sugerencia de categorizar mas la variable cancer_stage, y quitar los valores unknown que contiene esta por lo importante que es

In [ ]:
stage_map = {
    "In situ": "in_situ",
    "Localized only": "localized",
    "Regional by direct extension only": "regional",
    "Regional lymph nodes involved only": "regional",
    "Regional by both direct extension and lymph node involvement": "regional",
    "Distant site(s)/node(s) involved": "distant",
    "Unknown/unstaged/unspecified/DCO": "unknown"
}
df_raw["cancer_stage"] = df_raw["cancer_stage"].map(stage_map)

ordinal_stage_map = {
    "in_situ": 0,
    "localized": 1,
    "regional": 2,
    "distant": 3
}

df_raw["cancer_stage"] = df_raw["cancer_stage"].map(ordinal_stage_map)
df_raw = df_raw.dropna()

Sugerencia para convertir la variable sex en booleana

In [ ]:
df_raw["sex"] = df_raw["sex"].map({"Male": 1, "Female": 0}).astype(int)

Sugerencia para convertir la variable survival_months a categorica ordinal y encoded, pra que asi nuestro target tenga encoding

In [ ]:
df_raw["survival_risk"] = pd.qcut(df_raw["survival_months"], q=5, labels=[0, 1, 2, 3, 4])
df_raw["survival_risk"] = df_raw["survival_risk"].astype(int)
df_raw = df_raw.drop(columns=["survival_months"])

One hot encoding a variables categoricas

In [ ]:
df_raw = pd.get_dummies(df_raw,columns=["histologic_type_grouped", "primary_site_surgery", "tumor_grade", "cancer_site"], drop_first=True)

## 2. Función de pipeline completo para preprocesamiento

In [ ]:
def pipeline(datapath):
    df_raw = pd.read_csv("../data/raw/SEER_data.csv")
    columns_to_drop = [
        "Tumor Size Summary (2016+)",
        "Tumor Size Over Time Recode (1988+)",
        "Race recode (White, Black, Other)",
        "Summary stage 2000 (1998-2017)",
        "Primary Site - labeled",
        "SEER historic stage A (1973-2015)",
        "CS lymph nodes (2004-2015)",
        "CS mets at dx (2004-2015)",
        "Survival Days",
        "Diagnostic Confirmation",
        "Laterality",
        "Regional nodes examined (1988+)",
        "Behavior recode for analysis",
        "Reason no cancer-directed surgery",
        "SEER cause-specific death classification",
        "Vital status recode (study cutoff used)",
        "Radiation recode",
        "Regional nodes positive (1988+)",
        "CS extension (2004-2015)"
    ]
    df_raw = df_raw.drop(columns=columns_to_drop)

    new_column_names = [
        'age_group',
        'sex',
        'diagnosis_year',
        'histologic_type',
        'cancer_stage',
        'cancer_site',
        'tumor_grade',
        'bone_metastasis_at_dx',
        'brain_metastasis_at_dx',
        'liver_metastasis_at_dx',
        'lung_metastasis_at_dx',
        'primary_site_surgery',
        'received_chemotherapy',
        'survival_months',
    ]
    df_raw.columns = new_column_names

    df_raw = df_raw[df_raw["diagnosis_year"] >= 2010]

    df_raw = df_raw.dropna()
    df_raw = df_raw.drop_duplicates()
    dic_encoding = {"00 years":0, "01-04 years":1, "05-09 years": 2, "10-14 years": 3, "15-19 years": 4, "20-24 years": 5, "25-29 years": 6, "30-34 years": 7, "35-39 years": 8, "40-44 years": 9, "45-49 years": 10, "50-54 years": 11, "55-59 years": 12, "60-64 years": 13, "65-69 years": 14}
    df_raw["age_group"] = df_raw["age_group"].map(dic_encoding)
    
    df_raw = df_raw[df_raw["survival_months"] != "Unknown"]
    df_raw["survival_months"] = df_raw["survival_months"].astype(int)

    df_raw.loc[df_raw["primary_site_surgery"] == "Blank(s)", "primary_site_surgery"] = "Unknown"

    def map_surgery(code):
        if str(code) == "Unknown" or str(code) == "Blank(s)":
            return "Unknown"
        code = int(code)
        if code == 0:
            return "no_surgery"
        elif 1 <= code <= 29:
            return "minor_surgery"
        elif 30 <= code <= 59:
            return "partial_surgery"
        elif 60 <= code <= 89:
            return "radical_surgery"
        elif code == 90:
            return "surgery_nos"
        elif code == 98:
            return "not_applicable"
        elif code == 99:
            return "Unknown"
        else:
            print(code)
            return "other"  # safety fallback

    df_raw["primary_site_surgery"] = df_raw["primary_site_surgery"].apply(map_surgery)

    site_group_map = {
        # Cancer de cabeza y Cuello
        "Lip": "Head and Neck",
        "Tongue": "Head and Neck",
        "Salivary Gland": "Head and Neck",
        "Floor of Mouth": "Head and Neck",
        "Gum and Other Mouth": "Head and Neck",
        "Nasopharynx": "Head and Neck",
        "Tonsil": "Head and Neck",
        "Oropharynx": "Head and Neck",
        "Hypopharynx": "Head and Neck",
        "Other Oral Cavity and Pharynx": "Head and Neck",

        # CAncer de sistema digestivo
        "Esophagus": "Digestive",
        "Stomach": "Digestive",
        "Small Intestine": "Digestive",
        "Cecum": "Digestive",
        "Appendix": "Digestive",
        "Ascending Colon": "Digestive",
        "Hepatic Flexure": "Digestive",
        "Transverse Colon": "Digestive",
        "Splenic Flexure": "Digestive",
        "Descending Colon": "Digestive",
        "Sigmoid Colon": "Digestive",
        "Large Intestine, NOS": "Digestive",
        "Rectosigmoid Junction": "Digestive",
        "Rectum": "Digestive",
        "Anus, Anal Canal and Anorectum": "Digestive",
        "Liver": "Digestive",
        "Intrahepatic Bile Duct": "Digestive",
        "Gallbladder": "Digestive",
        "Other Biliary": "Digestive",
        "Pancreas": "Digestive",
        "Retroperitoneum": "Digestive",
        "Peritoneum, Omentum and Mesentery": "Digestive",
        "Other Digestive Organs": "Digestive",

        # Cancer respiratorio
        "Nose, Nasal Cavity and Middle Ear": "Respiratory",
        "Larynx": "Respiratory",
        "Lung and Bronchus": "Respiratory",
        "Pleura": "Respiratory",
        "Trachea, Mediastinum and Other Respiratory Organs": "Respiratory",

        # Otros tumores solidos
        "Bones and Joints": "Bone and Soft Tissue",
        "Soft Tissue including Heart": "Bone and Soft Tissue",
        "Melanoma of the Skin": "Skin",
        "Other Non-Epithelial Skin": "Skin",
        "Breast": "Breast",

        # Genitales femeninos
        "Cervix Uteri": "Female Genital",
        "Corpus Uteri": "Female Genital",
        "Uterus, NOS": "Female Genital",
        "Ovary": "Female Genital",
        "Vagina": "Female Genital",
        "Vulva": "Female Genital",
        "Other Female Genital Organs": "Female Genital",

        # Genitales masculinos
        "Prostate": "Male Genital",
        "Testis": "Male Genital",
        "Penis": "Male Genital",
        "Other Male Genital Organs": "Male Genital",

        # Urinarios
        "Urinary Bladder": "Urinary",
        "Kidney and Renal Pelvis": "Urinary",
        "Ureter": "Urinary",
        "Other Urinary Organs": "Urinary",

        # Otros sistemas mayores
        "Eye and Orbit": "Eye",
        "Brain and Other Nervous System": "Brain/CNS",
        "Thyroid": "Endocrine",
        "Other Endocrine including Thymus": "Endocrine",

        # Hematologicos o raros
        "Hodgkin - Nodal": "Hematologic",
        "Hodgkin - Extranodal": "Hematologic",
        "NHL - Nodal": "Hematologic",
        "NHL - Extranodal": "Hematologic",
        "Myeloma": "Hematologic",
        "Acute Lymphocytic Leukemia": "Hematologic",
        "Chronic Lymphocytic Leukemia": "Hematologic",
        "Other Lymphocytic Leukemia": "Hematologic",
        "Acute Myeloid Leukemia": "Hematologic",
        "Acute Monocytic Leukemia": "Hematologic",
        "Chronic Myeloid Leukemia": "Hematologic",
        "Other Myeloid/Monocytic Leukemia": "Hematologic",
        "Other Acute Leukemia": "Hematologic",
        "Aleukemic, subleukemic and NOS": "Hematologic",
        "Mesothelioma": "Hematologic",
        "Kaposi Sarcoma": "Hematologic",
        "Miscellaneous": "Hematologic",

        # No se sabe o no validos
        "Invalid": "Unknown/Invalid"
    }

    df_raw["cancer_site"] = df_raw["cancer_site"].map(site_group_map).fillna("Other/Unmapped")
    met_cols = [
        "bone_metastasis_at_dx",
        "brain_metastasis_at_dx",
        "liver_metastasis_at_dx",
        "lung_metastasis_at_dx"
    ]
    df_raw["metastasis_count"] = df_raw[met_cols].eq("Yes").sum(axis=1)
    df_raw["metastasis_count"].value_counts()
    df_raw = df_raw.drop(columns=["bone_metastasis_at_dx","brain_metastasis_at_dx", "liver_metastasis_at_dx", "lung_metastasis_at_dx"])

    top_histologies = df_raw["histologic_type"].value_counts().head(15).index
    df_raw["histologic_type_grouped"] = df_raw["histologic_type"].where(df_raw["histologic_type"].isin(top_histologies), "Other")

    df_raw["received_chemotherapy"] = df_raw["received_chemotherapy"].map({"Yes": 1, "No/Unknown": 0}).astype(int)

    stage_map = {
        "In situ": "in_situ",
        "Localized only": "localized",
        "Regional by direct extension only": "regional",
        "Regional lymph nodes involved only": "regional",
        "Regional by both direct extension and lymph node involvement": "regional",
        "Distant site(s)/node(s) involved": "distant",
        "Unknown/unstaged/unspecified/DCO": "unknown"
    }
    df_raw["cancer_stage"] = df_raw["cancer_stage"].map(stage_map)

    ordinal_stage_map = {
        "in_situ": 0,
        "localized": 1,
        "regional": 2,
        "distant": 3
    }

    df_raw["cancer_stage"] = df_raw["cancer_stage"].map(ordinal_stage_map)
    df_raw = df_raw.dropna()

    df_raw["sex"] = df_raw["sex"].map({"Male": 1, "Female": 0}).astype(int)
    df_raw["survival_risk"] = pd.qcut(df_raw["survival_months"], q=5, labels=[0, 1, 2, 3, 4])
    df_raw["survival_risk"] = df_raw["survival_risk"].astype(int)
    df_raw = df_raw.drop(columns=["survival_months"])
    df_raw = pd.get_dummies(df_raw,columns=["histologic_type_grouped", "primary_site_surgery", "tumor_grade", "cancer_site"], drop_first=True)
    df_raw.to_csv("../data/processed/processed_data.csv",index=False)

In [ ]:
data_path = "../data/raw/SEER_data.csv"
pipeline(data_path)

/tmp/ipykernel_204435/655151307.py:2: DtypeWarning: Columns (0: Regional nodes examined (1988+), 1: Regional nodes positive (1988+)) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("../data/raw/SEER_data.csv")


## 3. Conclusiones

Para esta tercera fase no hay conclusiones como tal, solo que se siguieron todas las sugerencias del EDA y se pudo construir una función pipeline que nos permite obtener la data preprocesada lista para modelar.